<font color="FF3B3B"><h1 align="left">Sistema multimodal para la detección e identificación de especies de hongos mediante visión por computador y modelos de lenguaje</h1></font>
<font color="#6E6E6E"><h2 align="left">Entrenamiento del modelo básico</h2></font>

#### David Alejandro Pedroza De Jesús

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shutil
import os
import tensorflow as tf
from google.colab import drive
from tensorflow.keras import layers, models
import shutil
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

In [ ]:
drive.mount('/content/drive/')
shutil.copy("/content/drive/MyDrive/kaggle.zip","/content/")
!unzip kaggle.zip
shutil.copy("/content/drive/MyDrive/InfoEspecies.csv","/content/")

# Carga de los datos

In [ ]:
rutas_val = pd.read_csv("kaggle/working/val.csv")
rutas_train = pd.read_csv("kaggle/working/train.csv")
rutas_test = pd.read_csv("kaggle/working/test.csv")
info_especies = pd.read_csv("InfoEspecies.csv")
info_especies = info_especies.drop(info_especies.columns[0], axis= "columns")

train = pd.merge(info_especies, rutas_train, on='label', how='inner')
test = pd.merge(info_especies, rutas_test, on='label', how='inner')
val = pd.merge(info_especies, rutas_val, on='label', how='inner')

In [ ]:
def ArreglarLasRutas(df):
    rutas_nuevas = []
    for path in df.image_path:
        path = path.lstrip("/")
        rutas_nuevas.append(path)
    df.image_path = rutas_nuevas

def Modificar_direc(df_rutas, subset):
    os.makedirs(f"Data/{subset}", exist_ok=True)

    ArreglarLasRutas(df_rutas)

    for path, label in zip(df_rutas.image_path, df_rutas.label):
        destino = f"Data/{subset}/{label}/"
        os.makedirs(destino, exist_ok=True)
        print(f"Copiando {path} --> {destino}")
        shutil.copy(path, destino)
        print(f"Terminado {label}")

In [ ]:
Modificar_direc(train, "train")
Modificar_direc(test, "test")
Modificar_direc(val, "valid")

In [ ]:
train_dir = "Data/train"
test_dir = "Data/test"
val_dir = "Data/valid"


train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    image_size=(64, 64),
    batch_size=32,
    label_mode='int'
)
test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    image_size=(64, 64),
    batch_size=32,
    label_mode='int'
)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    val_dir,
    image_size=(64, 64),
    batch_size=32,
    label_mode='int'
)

normalization_layer = layers.Rescaling(1./255)
train_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))
valid_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))

# Creamos el modelo

In [ ]:
num_classes = 57


model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(64,64,3)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(128, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(num_classes, activation='softmax')
])


model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(train_ds, epochs=20, validation_data=valid_ds)

test_loss, test_acc = model.evaluate(test_ds)
print(f"Precisión en test: {test_acc:.4f}")

# graficos error

In [ ]:
train_loss = history.history['loss']
val_loss = history.history['val_loss']

plt.plot(list(range(1,21)),train_loss, label='Training Loss')
plt.plot(list(range(1,21)),val_loss,label='Validation Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.show()

# matriz de confusión

In [ ]:
# Inicializar listas
y_true = []
y_pred = []

# Iterar sobre el dataset de prueba
for images, labels in test_ds:
    y_true.extend(labels.numpy())                        # Labels reales
    preds = model.predict(images)                        # Predicciones del modelo
    y_pred.extend(np.argmax(preds, axis=1))              # Clase con mayor probabilidad

# Convertir a arrays de numpy
y_true = np.array(y_true)
y_pred = np.array(y_pred)

In [ ]:
cm = confusion_matrix(y_true,y_pred ,normalize= "true").round(2)
plt.subplots(figsize=(60,50))
sns.heatmap(cm,xticklabels= info_especies.label.shape, yticklabels=info_especies.label.shape,annot=True)
plt.title("Matriz de confución")
plt.show()

In [ ]:
print(classification_report(y_true, y_pred))